# 07 Dashboard Export Assets

Purpose: prepare compact tables that the Streamlit dashboard can load quickly and reviewers can inspect.

In [ ]:
from __future__ import annotations

import json
import re
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", 120)

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

import sys
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.config import PROCESSED_DIR, TABLE_DIR, FIGURE_DIR, CPSC_DIR, PMPM_PROXY_REVENUE

for path in [PROCESSED_DIR, TABLE_DIR, FIGURE_DIR]:
    path.mkdir(parents=True, exist_ok=True)

print(f"Project root: {ROOT}")

In [ ]:
forecast = pd.read_csv(PROCESSED_DIR / "forecast_results.csv", parse_dates=["ds"])
metrics = pd.read_csv(TABLE_DIR / "forecast_metrics.csv")
state_growth = pd.read_csv(TABLE_DIR / "ma_scp_state_growth.csv")
pa = pd.read_csv(PROCESSED_DIR / "pa_predictions.csv")
delay = pd.read_csv(PROCESSED_DIR / "delay_predictions.csv")

best_models = metrics.sort_values(["target", "mape"]).groupby("target").head(1)
dashboard_forecast_summary = forecast[forecast["period_type"].isin(["holdout", "future"])].copy()
dashboard_state_opportunities = state_growth.head(30).copy()
dashboard_pa_summary = (
    pa.groupby(["procedure_type", "payer_type", "hybrid_risk_bucket"], as_index=False)
    .agg(cases=("hybrid_denial_risk", "size"), avg_denial_risk=("hybrid_denial_risk", "mean"))
    .sort_values("avg_denial_risk", ascending=False)
)
dashboard_delay_summary = (
    delay.groupby(["procedure_type", "payer_type", "risk_bucket"], as_index=False)
    .agg(combinations=("delay_risk_score", "size"), avg_delay_risk=("delay_risk_score", "mean"), delay_flags=("delay_flag", "sum"))
    .sort_values(["delay_flags", "avg_delay_risk"], ascending=False)
)

dashboard_forecast_summary.to_csv(TABLE_DIR / "dashboard_forecast_summary.csv", index=False)
dashboard_state_opportunities.to_csv(TABLE_DIR / "dashboard_state_opportunities.csv", index=False)
dashboard_pa_summary.to_csv(TABLE_DIR / "dashboard_pa_summary.csv", index=False)
dashboard_delay_summary.to_csv(TABLE_DIR / "dashboard_delay_summary.csv", index=False)
best_models.to_csv(TABLE_DIR / "dashboard_best_models.csv", index=False)

display(best_models)
display(dashboard_state_opportunities.head())
display(dashboard_pa_summary.head())
display(dashboard_delay_summary.head())

## Dashboard Asset Decision

The dashboard should load compact CSV summaries plus the forecast result table. The full long MA SCP table stays as Parquet for modeling, not dashboard rendering.